# 基本面綜合評分篩選 — 找出 ≥90 分的股票

**評分邏輯**（與 fundamental.html 一致，共 12 項，每項通過 +1）

| # | 欄位 | 說明 |
|---|------|------|
| 1 | 財報健康 | |
| 2 | 財報強勢 | |
| 3 | 財報動能強 | |
| 4 | EPS_前8季全為正 | |
| 5 | EPS_維持 | |
| 6 | 毛利率_維持 | |
| 7 | EPS_創新高 | |
| 8 | 毛利率_創新高 | |
| 9 | EPS_成長 | |
| 10 | EPS_YoY成長 | |
| 11 | 毛利率成長 | |
| 12 | 毛利率YoY成長 | |

分數 = pass數 / 12 × 100，≥90 分表示至少通過 11 項。

**資料來源**
- `Result/StockProfitability_all.xlsx`　→ 評分欄位取**最新一季**
- `Result/checkall/{date}.xlsx`　→ 補估值欄位（本益比、殖利率、淨值倍率、現價…）

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── 設定 ──────────────────────────────────────────────────────────
DATE         = '2026-05-28'          # ← 改這裡換日期
SCORE_CUTOFF = 70                    # ← 改這裡換門檻（0~100）

PROFIT_FILE  = Path('Result/StockProfitability_all.xlsx')
CHECKALL_FILE = Path(f'Result/checkall/{DATE}.xlsx')

# ── 評分欄位 ──────────────────────────────────────────────────────
SCORE_COLS = [
    '財報健康', '財報強勢', '財報動能強',
    'EPS_前8季全為正',
    'EPS_維持', '毛利率_維持',
    'EPS_創新高', '毛利率_創新高',
    'EPS_成長', 'EPS_YoY成長',
    '毛利率成長', '毛利率YoY成長',
]
N = len(SCORE_COLS)  # 12

print(f'日期：{DATE}　門檻：≥{SCORE_CUTOFF} 分')

日期：2026-05-28　門檻：≥70 分


In [2]:
# ── 1. 讀取財報歷史，取每支股票最新一季 ──────────────────────────
print('讀取 StockProfitability_all.xlsx ...')
df_profit = pd.read_excel(PROFIT_FILE, dtype={'stock_number': str})
df_profit['stock_number'] = df_profit['stock_number'].str.zfill(4)

# 最新一季（season_key 最大）
df_latest = (
    df_profit
    .sort_values('season_key')
    .groupby('stock_number', as_index=False)
    .last()
)

print(f'共 {len(df_latest)} 支股票，最新季範圍：{df_latest["季別"].min()} ~ {df_latest["季別"].max()}')

讀取 StockProfitability_all.xlsx ...
共 1965 支股票，最新季範圍：114.3Q ~ 115.1Q


In [3]:
# ── 2. 計算評分 ───────────────────────────────────────────────────
for col in SCORE_COLS:
    if col not in df_latest.columns:
        print(f'  ⚠️  欄位缺失：{col}，補 0')
        df_latest[col] = 0
    df_latest[col] = pd.to_numeric(df_latest[col], errors='coerce').fillna(0).astype(int)

df_latest['pass_count'] = df_latest[SCORE_COLS].sum(axis=1)
df_latest['score']      = (df_latest['pass_count'] / N * 100).round(1)

print('評分分布：')
print(df_latest['score'].describe().to_string())

評分分布：
count    1965.000000
mean       43.715623
std        27.423345
min         0.000000
25%        25.000000
50%        41.700000
75%        66.700000
max       100.000000


In [4]:
# ── 3. 讀取 checkall，補估值欄位 ─────────────────────────────────
print(f'讀取 {CHECKALL_FILE} ...')
df_check = pd.read_excel(CHECKALL_FILE, dtype={'stock_number': str})
df_check['stock_number'] = df_check['stock_number'].str.zfill(4)

# 要從 checkall 補的欄位
VAL_COLS = [
    'stock_number',
    'Type0',      # 股票名稱
    'Type1',      # 產業別
    'Type2',      # 上市/上櫃
    'now_price',  # 現價
    '_quote',     # 漲跌幅%
    '本益比', '同業平均本益比',
    '淨值倍率', '殖利率',
    '每股營收(元)',
    '營業收入-上月比較增減(%)',
    '營業收入-去年同月增減(%)',
    '累計營業收入-前期比較增減(%)',
    '總市值',
]
VAL_COLS_EXIST = [c for c in VAL_COLS if c in df_check.columns]
df_val = df_check[VAL_COLS_EXIST].copy()

print(f'checkall 共 {len(df_val)} 支股票')

讀取 Result\checkall\2026-05-28.xlsx ...
checkall 共 1918 支股票


In [5]:
# ── 4. Join ───────────────────────────────────────────────────────
df_score = df_latest[['stock_number','季別','EPS(元)','毛利率','EPS_MA_8','GM_MA_8',
                       'pass_count','score'] + SCORE_COLS].copy()

df_merged = df_score.merge(df_val, on='stock_number', how='left')

# 篩選 ≥ cutoff
df_high = df_merged[df_merged['score'] >= SCORE_CUTOFF].copy()
df_high = df_high.sort_values('score', ascending=False).reset_index(drop=True)

print(f'\n≥{SCORE_CUTOFF} 分的股票：{len(df_high)} 支')


≥70 分的股票：377 支


In [6]:
# ── 5. 展示結果 ───────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# 轉換估值欄位為數值
for col in ['本益比','同業平均本益比','淨值倍率','殖利率','now_price','EPS(元)','毛利率']:
    if col in df_high.columns:
        df_high[col] = pd.to_numeric(df_high[col], errors='coerce')

# 估值判斷
def pe_judge(row):
    pe, ipe = row.get('本益比'), row.get('同業平均本益比')
    if pd.isna(pe) or pd.isna(ipe) or ipe == 0: return '-'
    ratio = pe / ipe
    if ratio < 0.85: return '🟢 偏低'
    if ratio > 1.15: return '🔴 偏高'
    return '🟡 合理'

df_high['本益比判斷'] = df_high.apply(pe_judge, axis=1)

# 顯示欄位
SHOW = ['stock_number','Type0','Type1','score','pass_count',
        '季別','EPS(元)','毛利率',
        'now_price','本益比','同業平均本益比','本益比判斷','淨值倍率','殖利率',
        '累計營業收入-前期比較增減(%)']
SHOW = [c for c in SHOW if c in df_high.columns]

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 200)

df_high[SHOW]

,stock_number,Type0,Type1,score,pass_count,季別,EPS(元),毛利率,now_price,本益比,同業平均本益比,本益比判斷,淨值倍率,殖利率,累計營業收入-前期比較增減(%)
0,4576,NaN,NaN,100.00,12,115.1Q,0.96,39.01,NaN,NaN,NaN,-,NaN,NaN,NaN
1,2645,長榮航太,航運業,100.00,12,115.1Q,2.58,27.54,164.00,25.42,26.56,🟡 合理,4.85,3.00,12.75
2,6223,旺矽,半導體業,100.00,12,115.1Q,12.53,59.45,5915.00,158.68,109.43,🔴 偏高,36.78,0.37,42.51
3,3028,增你強,電子通路業,100.00,12,115.1Q,3.05,9.10,81.20,15.95,22.43,🟢 偏低,3.02,3.68,98.75
4,3026,禾伸堂,電子零組件業,100.00,12,115.1Q,2.86,24.24,605.00,82.60,78.50,🟡 合理,10.36,0.90,10.16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,3055,NaN,NaN,75.00,9,115.1Q,-0.15,31.88,NaN,NaN,NaN,-,NaN,NaN,NaN
373,3046,建碁,電腦及週邊設備業,75.00,9,115.1Q,1.20,9.07,54.40,14.20,34.15,🟢 偏低,2.77,5.15,16.58
374,6197,佳必琪,電子零組件業,75.00,9,115.1Q,3.21,35.77,282.50,28.98,78.50,🟢 偏低,7.56,2.36,16.04
375,3044,健鼎,電子零組件業,75.00,9,115.1Q,5.61,26.51,493.50,25.32,78.50,🟢 偏低,4.45,2.44,24.16


In [7]:
# ── 6. 每項通過細節 ───────────────────────────────────────────────
print(f'\n各評分項目通過率（≥{SCORE_CUTOFF} 分的 {len(df_high)} 支）：')
for col in SCORE_COLS:
    if col in df_high.columns:
        n = df_high[col].sum()
        print(f'  {col:<18} {int(n):>4} / {len(df_high)}  ({n/len(df_high)*100:.0f}%)')


各評分項目通過率（≥70 分的 377 支）：
  財報健康                377 / 377  (100%)
  財報強勢                167 / 377  (44%)
  財報動能強               376 / 377  (100%)
  EPS_前8季全為正          223 / 377  (59%)
  EPS_維持              377 / 377  (100%)
  毛利率_維持              377 / 377  (100%)
  EPS_創新高             244 / 377  (65%)
  毛利率_創新高             261 / 377  (69%)
  EPS_成長              351 / 377  (93%)
  EPS_YoY成長           377 / 377  (100%)
  毛利率成長               351 / 377  (93%)
  毛利率YoY成長            376 / 377  (100%)


In [8]:
# ── 7. 產業分布 ───────────────────────────────────────────────────
if 'Type1' in df_high.columns:
    print(f'\n產業分布（≥{SCORE_CUTOFF} 分）：')
    print(df_high['Type1'].value_counts().to_string())


產業分布（≥70 分）：
半導體業        64
電子零組件業      42
電腦及週邊設備業    22
生技醫療業       21
其他電子業       21
光電業         18
鋼鐵工業        14
通信網路業       14
化學工業        13
電子通路業       12
電機機械        12
其他          12
綠能環保        11
航運業          9
數位雲端         8
觀光事業         8
塑膠工業         7
電器電纜         7
紡織纖維         7
資訊服務業        6
建材營造         5
食品工業         5
油電燃氣業        4
貿易百貨         4
橡膠工業         4
金融業          3
居家生活         3
文化創意業        3
汽車工業         1
金融保險         1
玻璃陶瓷         1
造紙工業         1
運動休閒         1


In [9]:
# ── 8. 輸出 Excel ─────────────────────────────────────────────────
out_path = Path(f'Result/fundamental_score_{DATE}_top{SCORE_CUTOFF}.xlsx')
out_path.parent.mkdir(parents=True, exist_ok=True)

# 輸出完整欄位
df_high.to_excel(out_path, index=False)
print(f'已輸出：{out_path}　（{len(df_high)} 支股票）')

已輸出：Result\fundamental_score_2026-05-28_top70.xlsx　（377 支股票）


In [10]:
# ── 9. 不同門檻的分布（參考用）──────────────────────────────────
print('各分數門檻對應股票數：')
for cutoff in [100, 90, 80, 70, 60, 50]:
    n = (df_merged['score'] >= cutoff).sum()
    bar = '█' * (n // 10)
    print(f'  ≥{cutoff:3d} 分：{n:4d} 支  {bar}')

各分數門檻對應股票數：
  ≥100 分：  78 支  ███████
  ≥ 90 分： 166 支  ████████████████
  ≥ 80 分： 220 支  ██████████████████████
  ≥ 70 分： 377 支  █████████████████████████████████████
  ≥ 60 分： 516 支  ███████████████████████████████████████████████████
  ≥ 50 分： 878 支  ███████████████████████████████████████████████████████████████████████████████████████


In [11]:
print(df_high[SHOW]['stock_number'].tolist())


['4576', '2645', '6223', '3028', '3026', '6274', '3017', '6417', '2762', '6509', '2753', '6584', '2707', '2633', '6206', '6651', '2610', '6721', '2478', '2467', '6753', '2451', '6757', '6805', '2423', '2377', '3037', '6173', '2330', '3577', '4766', '4721', '1233', '4904', '4111', '4106', '3716', '5234', '5236', '3702', '5289', '5340', '6151', '5386', '3479', '3388', '5536', '5538', '6005', '3305', '3293', '3264', '3260', '3167', '6907', '6510', '8390', '8299', '1503', '8086', '8091', '8028', '8021', '1590', '1773', '7765', '1733', '8358', '8341', '2027', '1618', '1268', '2308', '9937', '8271', '8210', '7750', '1726', '3224', '8473', '3555', '3550', '3135', '5351', '3522', '8423', '3508', '5468', '6015', '5475', '6016', '1586', '3372', '7610', '3685', '3623', '4971', '9944', '4807', '4711', '4569', '1303', '4923', '4939', '4502', '4949', '4956', '4426', '4967', '4147', '3632', '4973', '4989', '4991', '5228', '1313', '9929', '1326', '8935', '8489', '1447', '3691', '3122', '6150', '8131',

In [12]:
print(df_high[df_high['EPS_前8季全為正']==1]['stock_number'].tolist())

['4576', '2645', '6223', '3028', '3026', '6274', '3017', '6417', '2762', '6509', '2753', '6584', '2707', '2633', '6206', '6651', '2610', '6721', '2478', '2467', '6753', '2451', '6757', '6805', '2423', '2377', '3037', '6173', '2330', '3577', '4766', '4721', '1233', '4904', '4111', '4106', '3716', '5234', '5236', '3702', '5289', '5340', '6151', '5386', '3479', '3388', '5536', '5538', '6005', '3305', '3293', '3264', '3260', '3167', '6907', '6510', '8390', '8299', '1503', '8086', '8091', '8028', '8021', '1590', '1773', '7765', '1733', '8358', '8341', '2027', '1618', '1268', '2308', '9937', '8271', '8210', '7750', '1726', '7820', '6908', '6841', '7711', '6811', '6903', '6894', '7768', '7556', '6727', '5471', '6023', '6196', '6215', '5607', '6277', '5478', '8415', '5434', '6712', '4749', '6570', '8438', '5276', '6640', '5274', '5243', '5285', '1216', '9958', '3045', '1623', '1709', '1717', '2404', '1788', '2049', '3467', '3014', '2368', '3357', '3324', '2108', '2316', '3090', '2441', '3645',